# Minimax With Alpha-Beta Pruning: Tic-Tac-Toe Bot

Minimax is a game AI algorithm. It looks ahead at possible future moves and assumes both players will play as well as they can.

The idea comes from game theory, especially work by John von Neumann on optimal play in adversarial settings. Minimax and alpha-beta pruning shaped early chess programs and still teach the core logic behind strategic search.

Alpha-beta pruning is the speed-up: it stops searching branches that cannot change the final decision.

In this notebook, you will build an invincible Tic-Tac-Toe bot and watch it prune useless branches.

<details>
<summary>Big idea</summary>

The bot asks: "If I move here, and my opponent replies perfectly, what is the best outcome I can force?"

</details>

## 1. The Mental Model

Tic-Tac-Toe has two kinds of turns:

- **Max player**: tries to get the highest score
- **Min player**: tries to force the lowest score

Scores for the bot:

- win: positive
- draw: `0`
- loss: negative

Alpha-beta keeps two guardrails:

- **alpha**: the best score Max can already guarantee
- **beta**: the best score Min can already guarantee

<details>
<summary>Pruning hint</summary>

If a branch is already worse than an option found earlier, the bot stops exploring it. Perfect play does not need wasted searching.

</details>

## 2. Build the Objects

Implementation plan:

1. `TicTacToeBoard` stores the board and legal moves.
2. `SearchSnapshot` records one search decision.
3. `SearchResult` stores the best move and search stats.
4. `MinimaxBot` recursively scores future game states.
5. `SearchReplay` prints the search and pruning moments.
6. `TicTacToeMatch` lets you play a scripted game against the bot.

<details>
<summary>Implementation hint</summary>

Minimax is recursion over a game tree. Each node is a board. Each edge is a move. Leaves are wins, losses, or draws.

</details>

**Example state.** Create `WIN_LINES`, the concrete values used in the next run.


In [ ]:
from dataclasses import dataclass

from math import inf

WIN_LINES = (
    (0, 1, 2),
    (3, 4, 5),
    (6, 7, 8),
    (0, 3, 6),
    (1, 4, 7),
    (2, 5, 8),
    (0, 4, 8),
    (2, 4, 6),
)


**Object model.** Define `TicTacToeBoard`, the named objects used by the next examples.


In [ ]:
@dataclass(frozen=True)
class TicTacToeBoard:
    cells: tuple[str, ...] = (" ", " ", " ", " ", " ", " ", " ", " ", " ")

    def available_moves(self) -> list[int]:
        return [index for index, marker in enumerate(self.cells) if marker == " "]

    def place(self, move: int, marker: str) -> "TicTacToeBoard":
        if move not in self.available_moves():
            raise ValueError(f"Move {move + 1} is not available.")

        new_cells = list(self.cells)
        new_cells[move] = marker
        return TicTacToeBoard(tuple(new_cells))

    def winner(self) -> str | None:
        for first, second, third in WIN_LINES:
            marker = self.cells[first]
            if marker != " " and marker == self.cells[second] == self.cells[third]:
                return marker
        return None

    def is_full(self) -> bool:
        return " " not in self.cells

    def is_terminal(self) -> bool:
        return self.winner() is not None or self.is_full()

    def render(self, show_positions: bool = False) -> str:
        display_cells = []
        for index, marker in enumerate(self.cells):
            if marker == " " and show_positions:
                display_cells.append(str(index + 1))
            elif marker == " ":
                display_cells.append(".")
            else:
                display_cells.append(marker)

        rows = [" | ".join(display_cells[start:start + 3]) for start in range(0, 9, 3)]
        return "\n---------\n".join(rows)


**Trace model.** Define `SearchSnapshot`, `SearchResult`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass(frozen=True)
class SearchSnapshot:
    depth: int
    player: str
    move: int | None
    alpha: float
    beta: float
    score: int | None
    action: str
    board: TicTacToeBoard
    note: str

@dataclass(frozen=True)
class SearchResult:
    move: int | None
    score: int
    nodes_visited: int
    pruned_branches: int
    snapshots: tuple[SearchSnapshot, ...]


**Algorithm engine.** Define `MinimaxBot`, the class that runs the main simulation or algorithm.


In [ ]:
class MinimaxBot:
    def __init__(self, marker: str = "O", opponent: str = "X", use_pruning: bool = True):
        self.marker = marker
        self.opponent = opponent
        self.use_pruning = use_pruning
        self.snapshots: list[SearchSnapshot] = []
        self.nodes_visited = 0
        self.pruned_branches = 0

    def choose_move(self, board: TicTacToeBoard) -> SearchResult:
        self.snapshots = []
        self.nodes_visited = 0
        self.pruned_branches = 0

        best_score = -inf
        best_move = None
        alpha = -inf
        beta = inf

        for move in self._ordered_moves(board):
            next_board = board.place(move, self.marker)
            score = self._minimax(next_board, self.opponent, depth=1, alpha=alpha, beta=beta)
            self.snapshots.append(
                SearchSnapshot(
                    depth=0,
                    player=self.marker,
                    move=move,
                    alpha=alpha,
                    beta=beta,
                    score=score,
                    action="consider",
                    board=next_board,
                    note=f"Bot tests move {move + 1} and gets score {score}.",
                )
            )

            if score > best_score:
                best_score = score
                best_move = move

            alpha = max(alpha, best_score)

        return SearchResult(
            move=best_move,
            score=int(best_score),
            nodes_visited=self.nodes_visited,
            pruned_branches=self.pruned_branches,
            snapshots=tuple(self.snapshots),
        )

    def _ordered_moves(self, board: TicTacToeBoard) -> list[int]:
        preferred_order = [4, 0, 2, 6, 8, 1, 3, 5, 7]
        available = set(board.available_moves())
        return [move for move in preferred_order if move in available]

    def _other_player(self, marker: str) -> str:
        return self.opponent if marker == self.marker else self.marker

    def _terminal_score(self, board: TicTacToeBoard, depth: int) -> int | None:
        winner = board.winner()
        if winner == self.marker:
            return 10 - depth
        if winner == self.opponent:
            return depth - 10
        if board.is_full():
            return 0
        return None

    def _minimax(self, board: TicTacToeBoard, current_player: str, depth: int, alpha: float, beta: float) -> int:
        self.nodes_visited += 1
        terminal_score = self._terminal_score(board, depth)
        if terminal_score is not None:
            self.snapshots.append(
                SearchSnapshot(depth, current_player, None, alpha, beta, terminal_score, "leaf", board, "Reached a finished board.")
            )
            return terminal_score

        if current_player == self.marker:
            best_score = -inf
            for move in self._ordered_moves(board):
                next_board = board.place(move, current_player)
                score = self._minimax(next_board, self._other_player(current_player), depth + 1, alpha, beta)
                best_score = max(best_score, score)
                alpha = max(alpha, best_score)

                if self.use_pruning and beta <= alpha:
                    self.pruned_branches += 1
                    self.snapshots.append(
                        SearchSnapshot(depth, current_player, move, alpha, beta, int(best_score), "prune", next_board, "Max already has a better path elsewhere.")
                    )
                    break

            return int(best_score)

        best_score = inf
        for move in self._ordered_moves(board):
            next_board = board.place(move, current_player)
            score = self._minimax(next_board, self._other_player(current_player), depth + 1, alpha, beta)
            best_score = min(best_score, score)
            beta = min(beta, best_score)

            if self.use_pruning and beta <= alpha:
                self.pruned_branches += 1
                self.snapshots.append(
                    SearchSnapshot(depth, current_player, move, alpha, beta, int(best_score), "prune", next_board, "Min already has a better path elsewhere.")
                )
                break

        return int(best_score)


## 3. Ask the Bot for a Move

Here is a midgame board where `O` is the bot. The bot searches future moves and chooses the best one under perfect play.

<details>
<summary>Board positions</summary>

Cells are numbered left to right, top to bottom: `1` through `9`. Empty cells are shown as `.`.

</details>

In [2]:
example_board = TicTacToeBoard((
    "X", "O", "X",
    " ", "O", " ",
    " ", "X", " ",
))

bot = MinimaxBot(marker="O", opponent="X", use_pruning=True)
decision = bot.choose_move(example_board)
chosen_board = example_board.place(decision.move, bot.marker)

print("Current board:")
print(example_board.render())
print(f"\nBot chooses position {decision.move + 1} with score {decision.score}.")
print(f"Nodes visited: {decision.nodes_visited}")
print(f"Pruned branches: {decision.pruned_branches}")
print("\nBoard after bot move:")
print(chosen_board.render())

Current board:
X | O | X
---------
. | O | .
---------
. | X | .

Bot chooses position 7 with score 0.
Nodes visited: 46
Pruned branches: 17

Board after bot move:
X | O | X
---------
. | O | .
---------
O | X | .


## 4. Replay the Search

The bot records snapshots while searching. We will print a few top-level move tests and the first pruning moments.

<details>
<summary>Reading alpha and beta</summary>

When alpha catches beta, the current branch cannot improve the final choice, so alpha-beta stops exploring that branch.

</details>

In [3]:
class SearchReplay:
    def __init__(self, result: SearchResult):
        self.result = result

    def show_top_choices(self) -> None:
        print("Top-level choices:")
        for snapshot in self.result.snapshots:
            if snapshot.action == "consider":
                print(f"  move {snapshot.move + 1}: score {snapshot.score}")

    def show_prunes(self, limit: int = 5) -> None:
        prunes = [snapshot for snapshot in self.result.snapshots if snapshot.action == "prune"]
        print(f"\nFirst {min(limit, len(prunes))} pruning snapshots:")
        for snapshot in prunes[:limit]:
            move_text = snapshot.move + 1 if snapshot.move is not None else "none"
            print(f"depth={snapshot.depth} player={snapshot.player} move={move_text} alpha={snapshot.alpha} beta={snapshot.beta}")
            print(f"  {snapshot.note}")
            print(snapshot.board.render())
            print()


SearchReplay(decision).show_top_choices()
SearchReplay(decision).show_prunes(limit=4)

Top-level choices:
  move 7: score 0
  move 9: score 0
  move 4: score 0
  move 6: score 0

First 4 pruning snapshots:
depth=2 player=O move=9 alpha=0 beta=0
  Max already has a better path elsewhere.
X | O | X
---------
X | O | .
---------
O | X | O

depth=2 player=O move=9 alpha=0 beta=0
  Max already has a better path elsewhere.
X | O | X
---------
. | O | X
---------
O | X | O

depth=3 player=X move=6 alpha=0 beta=0
  Min already has a better path elsewhere.
X | O | X
---------
O | O | X
---------
X | X | O

depth=3 player=X move=4 alpha=0 beta=-6
  Min already has a better path elsewhere.
X | O | X
---------
X | O | O
---------
X | X | O



## 5. Play a Scripted Game Against the Bot

This notebook version uses a move list instead of keyboard input, so the whole game runs safely in one cell. Change the human move list and rerun to try different games.

<details>
<summary>Invincible bot rule</summary>

With perfect minimax play, Tic-Tac-Toe is at worst a draw for the bot. If the human makes a mistake, the bot can win.

</details>

**Object model.** Define `GameTurn`, the named objects used by the next examples.


In [ ]:
@dataclass(frozen=True)
class GameTurn:
    actor: str
    move: int
    board: TicTacToeBoard
    note: str


**Object model.** Define `TicTacToeMatch`, the named objects used by the next examples.


In [ ]:
class TicTacToeMatch:
    def __init__(self, bot: MinimaxBot):
        self.bot = bot

    def _safe_human_move(self, board: TicTacToeBoard, planned_position: int) -> tuple[int, str]:
        planned_move = planned_position - 1
        if planned_move in board.available_moves():
            return planned_move, f"Human used planned position {planned_position}."

        fallback_move = board.available_moves()[0]
        return fallback_move, f"Position {planned_position} was taken, so human used {fallback_move + 1}."

    def play_script(self, human_positions: list[int]) -> list[GameTurn]:
        board = TicTacToeBoard()
        turns: list[GameTurn] = []

        for planned_position in human_positions:
            if board.is_terminal():
                break

            human_move, human_note = self._safe_human_move(board, planned_position)
            board = board.place(human_move, self.bot.opponent)
            turns.append(GameTurn("Human X", human_move + 1, board, human_note))

            if board.is_terminal():
                break

            bot_decision = self.bot.choose_move(board)
            board = board.place(bot_decision.move, self.bot.marker)
            turns.append(
                GameTurn(
                    "Bot O",
                    bot_decision.move + 1,
                    board,
                    f"searched {bot_decision.nodes_visited} nodes and pruned {bot_decision.pruned_branches} branch(es)",
                )
            )

        return turns


**Trace model.** Define `MatchReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class MatchReplay:
    def __init__(self, turns: list[GameTurn]):
        self.turns = turns

    def show(self) -> None:
        for turn_number, turn in enumerate(self.turns, start=1):
            print(f"Turn {turn_number}: {turn.actor} chose {turn.move}")
            print(f"  {turn.note}")
            print(turn.board.render())
            print()

        final_board = self.turns[-1].board if self.turns else TicTacToeBoard()
        winner = final_board.winner()
        if winner is None:
            print("Result: draw")
        else:
            print(f"Result: {winner} wins")

human_positions = [1, 9, 6, 7, 2]

match_bot = MinimaxBot(marker="O", opponent="X", use_pruning=True)

turns = TicTacToeMatch(match_bot).play_script(human_positions)

MatchReplay(turns).show()


## 6. Experiment: Plain Minimax vs Alpha-Beta

Both versions choose the same move because they reason about the same game tree. Alpha-beta gets there by visiting fewer nodes.

<details>
<summary>Optimization hint</summary>

Alpha-beta pruning does not change the answer. It only avoids branches that are provably irrelevant.

</details>

In [5]:
comparison_board = TicTacToeBoard((
    "X", " ", " ",
    " ", "O", " ",
    " ", " ", "X",
))

plain_bot = MinimaxBot(marker="O", opponent="X", use_pruning=False)
pruned_bot = MinimaxBot(marker="O", opponent="X", use_pruning=True)

plain_result = plain_bot.choose_move(comparison_board)
pruned_result = pruned_bot.choose_move(comparison_board)

print("Comparison board:")
print(comparison_board.render())
print()
print(f"Plain minimax:      move={plain_result.move + 1}, score={plain_result.score}, nodes={plain_result.nodes_visited}, prunes={plain_result.pruned_branches}")
print(f"Alpha-beta pruning: move={pruned_result.move + 1}, score={pruned_result.score}, nodes={pruned_result.nodes_visited}, prunes={pruned_result.pruned_branches}")
print(f"Same answer: {plain_result.move == pruned_result.move and plain_result.score == pruned_result.score}")

Comparison board:
X | . | .
---------
. | O | .
---------
. | . | X

Plain minimax:      move=2, score=0, nodes=1052, prunes=0
Alpha-beta pruning: move=2, score=0, nodes=319, prunes=104
Same answer: True


## What You Should Remember

Minimax is decision-making by perfect lookahead:

- The bot scores terminal futures: win, draw, or loss.
- Max turns choose the highest score.
- Min turns choose the lowest score.
- Recursion explores the game tree.
- Alpha-beta pruning skips branches that cannot change the answer.
- In Tic-Tac-Toe, perfect minimax play makes the bot impossible to beat.

<details>
<summary>Where this shows up</summary>

Minimax-style search appears in game AI, adversarial planning, decision theory, chess engines, checkers, Tic-Tac-Toe, Connect Four, and any turn-based setting with competing goals.

</details>

## Visual Trace + Rigor Studio

**Problem frame.** Choose moves against an opponent who also chooses optimally.

**Interactive animation target.** Animate game-tree expansion and value backup from leaves to root.

**Correctness handle.** A max node takes the best child value; a min node takes the worst child value for max.

**Complexity handle.** O(b^d) for branching factor b and depth d before pruning or heuristics.

**Failure mode to test.** Wrong evaluation functions make limited-depth search confidently choose bad moves.

**Studio task.** Change one leaf value and trace whether the root decision changes.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware import AlgorithmPlayer, AlgorithmTrace, TraceStep, render_trace_table

# Convert the implementation above into snapshots:
# trace = AlgorithmTrace("Topic trace")
# trace.append("start", {"your_state": ...}, "What changed?", invariant="What remains true?")
# AlgorithmPlayer(trace, your_renderer).display()
print("Use AlgorithmTrace to turn this notebook's algorithm into a step-by-step visual player.")
